# Spark Basics on a Real Cluster

This notebook is the **driver**. It connects to the Spark master (`spark://spark-master:7077`),
which gives it **executors** on the 3 worker containers. This is **client mode**: the driver runs here, in Jupyter.

Useful pages while you work:
- Master UI (workers, running apps): http://localhost:8080
- Application UI (jobs, stages, tasks): http://localhost:4040 (only while the SparkSession below is running)

## 1. Connect to the cluster

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-basics-notebook")
    .master("spark://spark-master:7077")        # the master container
    .config("spark.driver.host", "spark-jupyter")  # executors connect back to this notebook
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version :", spark.version)
print("Master        :", sc.master)
print("Parallelism   :", sc.defaultParallelism, "(total executor cores)")

Spark version : 4.0.0
Master        : spark://spark-master:7077
Parallelism   : 6 (total executor cores)


Open http://localhost:8080 now: you should see **spark-basics-notebook** under *Running Applications*,
using cores on the 3 workers.

## 2. Create a DataFrame

In [3]:
data = [
    ("Alice", "Engineering", 120000, 2019),
    ("Bob",   "Engineering",  95000, 2021),
    ("Cara",  "Marketing",    70000, 2020),
    ("Dan",   "Marketing",    82000, 2018),
    ("Eve",   "Sales",        60000, 2022),
    ("Frank", "Sales",        65000, 2017),
]
df = spark.createDataFrame(data, ["name", "dept", "salary", "joined"])

df.printSchema()
df.show()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- joined: long (nullable = true)



+-----+-----------+------+------+
| name|       dept|salary|joined|
+-----+-----------+------+------+
|Alice|Engineering|120000|  2019|
|  Bob|Engineering| 95000|  2021|
| Cara|  Marketing| 70000|  2020|
|  Dan|  Marketing| 82000|  2018|
|  Eve|      Sales| 60000|  2022|
|Frank|      Sales| 65000|  2017|
+-----+-----------+------+------+



## 3. Transformations are lazy, actions run jobs

`filter`, `withColumn`, `select` are **transformations**: Spark only records a plan.
`show`, `count`, `collect`, `write` are **actions**: they actually run a job on the executors.

In [4]:
# Transformations: nothing runs yet
high_paid = (
    df.filter(F.col("salary") > 70000)
      .withColumn("salary_k", F.col("salary") / 1000)
      .select("name", "dept", "salary_k")
)

# See the plan Spark built
high_paid.explain()

== Physical Plan ==
*(1) Project [name#0, dept#1, (cast(salary#2L as double) / 1000.0) AS salary_k#17]
+- *(1) Filter (isnotnull(salary#2L) AND (salary#2L > 70000))
   +- *(1) Scan ExistingRDD[name#0,dept#1,salary#2L,joined#3L]




In [5]:
# Action: now a job runs (check the Jobs tab at http://localhost:4040)
high_paid.show()

+-----+-----------+--------+
| name|       dept|salary_k|
+-----+-----------+--------+
|Alice|Engineering|   120.0|
|  Bob|Engineering|    95.0|
|  Dan|  Marketing|    82.0|
+-----+-----------+--------+



## 4. Aggregations

In [6]:
(
    df.groupBy("dept")
      .agg(
          F.count("*").alias("people"),
          F.avg("salary").alias("avg_salary"),
          F.max("salary").alias("max_salary"),
      )
      .orderBy(F.desc("avg_salary"))
      .show()
)

+-----------+------+----------+----------+
|       dept|people|avg_salary|max_salary|
+-----------+------+----------+----------+
|Engineering|     2|  107500.0|    120000|
|  Marketing|     2|   76000.0|     82000|
|      Sales|     2|   62500.0|     65000|
+-----------+------+----------+----------+



## 5. Spark SQL

Register a DataFrame as a view and query it with plain SQL.

In [ ]:
df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT dept,
           COUNT(*)            AS people,
           ROUND(AVG(salary))  AS avg_salary
    FROM employees
    WHERE joined >= 2018
    GROUP BY dept
    ORDER BY avg_salary DESC
""").show()

## 6. See the work spread across workers

A bigger dataset split into **12 partitions**. Each partition is processed by a task on some executor.
The code below asks every partition *which container it ran on*.

In [ ]:
import socket

big = spark.range(0, 10_000_000, numPartitions=12)
print("Rows      :", big.count())
print("Partitions:", big.rdd.getNumPartitions())

def where_am_i(rows):
    n = sum(1 for _ in rows)
    yield (socket.gethostname(), n)

per_host = (
    big.rdd.mapPartitions(where_am_i)
       .toDF(["worker_container", "rows"])
       .groupBy("worker_container")
       .agg(F.count("*").alias("partitions"), F.sum("rows").alias("rows"))
)
per_host.show(truncate=False)

## 7. Write and read files

`/opt/spark/work-dir` is the `workspace/` folder on your Mac, mounted into **every** container,
so all executors can write to it. Look in `workspace/output/` on your Mac afterwards.

In [ ]:
out = "/opt/spark/work-dir/output/employees_parquet"

df.write.mode("overwrite").partitionBy("dept").parquet(out)

back = spark.read.parquet(out)
back.show()
print("Files are partitioned by dept: see workspace/output/employees_parquet/ on your Mac")

In [ ]:
csv_out = "/opt/spark/work-dir/output/employees_csv"
df.coalesce(1).write.mode("overwrite").option("header", True).csv(csv_out)

spark.read.option("header", True).option("inferSchema", True).csv(csv_out).show()

## 8. From Spark to pandas

`toPandas()` collects everything to the driver, so only use it on small results.

In [ ]:
pdf = df.groupBy("dept").agg(F.avg("salary").alias("avg_salary")).toPandas()
pdf

## 9. Stop the session

This releases the executors on the workers. The app moves to *Completed Applications* at http://localhost:8080.

In [ ]:
spark.stop()